# Intermediate SQL

In this session we will work with subqueries, joins and other concepts discussed in the lecture. SQL can be used within the Python Environment itself.

We will be using the sqlite3 package in Python. The documentation can be found here: https://docs.python.org/3/library/sqlite3.html

We will use the SAKILA database which is available from Github from the following link: https://github.com/bradleygrant/sakila-sqlite3/tree/main

The Sakila database is a nicely normalised database modelling a DVD rental store. The documentation can be found in the following link: https://dev.mysql.com/doc/sakila/en/sakila-introduction.html

#Dataset Preparation:

1. Download the `sakila_master.database` from the github link above by clicking on `Ready-To-Use SQLite3 Database File`.
2. Upload the file into Google Colab if you are using google colab. 
3. Import packages:
`pandas, sqlite3, numpy, matplotlib`

In [1]:
# Importing Packages:

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import sqlite3
import matplotlib.pyplot as plt

In [ ]:
# run this if using google colab
### Mounting google drive ###
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# run this if using google colab
### Pathname of the database file to be opened ###
path = "/content/drive/MyDrive/"  #Insert path here
database = path + 'sakila_master.db'

In [2]:
## if using Jupyter Notebook
## first download the database and put it in the same folder as this Jupyter notebook
## then run this cell
## Path to your SQLite database file (same folder as Jupyter notebook)
database = "sakila_master.db"  

# Create or Load SQLite database:

connect() opens a connection to the SQLite database file database. By default returns a Connection object, unless a custom factory is given.

Syntax: `sqlite3.connect(database[, timeout, detect_types, isolation_level, check_same_thread, factory, cached_statements, uri])`

where, database -> database is a path-like object which is pathname of the database file to be opened.

In [3]:
### Connecting to database and checking tables ###

connection_db = sqlite3.connect(database)

tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_db)
tables  
# the SQLite database file is made up of a sequence of pages
# the rootpage is the page number where the main structure of a table starts
# name: actual name of the object (e.g, a table, an index)
# tbl_name: the name of the table that the object belongs to

,type,name,tbl_name,rootpage,sql
0,table,actor,actor,2,CREATE TABLE actor (\n actor_id numeric NOT N...
1,table,country,country,5,CREATE TABLE country (\n country_id SMALLINT ...
2,table,city,city,9,"CREATE TABLE city (\n city_id int NOT NULL,\n..."
3,table,address,address,13,CREATE TABLE address (\n address_id int NOT N...
4,table,language,language,17,CREATE TABLE language (\n language_id SMALLIN...
5,table,category,category,19,CREATE TABLE category (\n category_id SMALLIN...
6,table,customer,customer,22,CREATE TABLE customer (\n customer_id INT NOT...
7,table,film,film,29,"CREATE TABLE film (\n film_id int NOT NULL,\n..."
8,table,film_actor,film_actor,35,CREATE TABLE film_actor (\n actor_id INT NOT ...
9,table,film_category,film_category,40,CREATE TABLE film_category (\n film_id INT NO...


# SQL DDL Commands:


## ALTER TABLE:

Here we will use the ` ALTER TABLE` command with `RENAME` to show how we can use them in conjunction.

In [4]:
cursor = connection_db.execute('''
                                  ALTER TABLE language
                                  RENAME TO movie_language;
                                  ''')

In [5]:
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_db)
tables

,type,name,tbl_name,rootpage,sql
0,table,actor,actor,2,CREATE TABLE actor (\n actor_id numeric NOT N...
1,table,country,country,5,CREATE TABLE country (\n country_id SMALLINT ...
2,table,city,city,9,"CREATE TABLE city (\n city_id int NOT NULL,\n..."
3,table,address,address,13,CREATE TABLE address (\n address_id int NOT N...
4,table,movie_language,movie_language,17,"CREATE TABLE ""movie_language"" (\n language_id..."
5,table,category,category,19,CREATE TABLE category (\n category_id SMALLIN...
6,table,customer,customer,22,CREATE TABLE customer (\n customer_id INT NOT...
7,table,film,film,29,"CREATE TABLE film (\n film_id int NOT NULL,\n..."
8,table,film_actor,film_actor,35,CREATE TABLE film_actor (\n actor_id INT NOT ...
9,table,film_category,film_category,40,CREATE TABLE film_category (\n film_id INT NO...


Similarly other DDL commands can be used with the `cursor` and `conenction_db.execute`.

# SELECT statements with added complexity:

## SELECT DISTINCT:

This selects only the distinct instances from the table:

In [6]:
# Without distinct:

tables = pd.read_sql("""SELECT first_name
                        FROM actor
                        WHERE first_name='ED';""", connection_db)
tables

,first_name
0,ED
1,ED
2,ED


In [7]:
# With distinct:

tables = pd.read_sql("""SELECT DISTINCT first_name
                        FROM actor
                        WHERE first_name='ED';""", connection_db)
tables

,first_name
0,ED


Keeps only one instance of `first_name` = "ED"

## SELECT TOP:

Selects TOP number of rows. Need to specify number. Same as LIMIT.

`SQLITE` does not have the `TOP` keyword. You can use `LIMIT` instead.

In [8]:
# With distinct:

tables = pd.read_sql("""SELECT *
                        FROM actor
                        LIMIT 5;""", connection_db)
tables

,actor_id,first_name,last_name,last_update
0,1,PENELOPE,GUINESS,2020-12-23 07:12:29
1,2,NICK,WAHLBERG,2020-12-23 07:12:29
2,3,ED,CHASE,2020-12-23 07:12:29
3,4,JENNIFER,DAVIS,2020-12-23 07:12:29
4,5,JOHNNY,LOLLOBRIGIDA,2020-12-23 07:12:29


In [9]:
## double check
tables = pd.read_sql("""SELECT *
                        FROM actor
                        ;""", connection_db)
tables.head()

,actor_id,first_name,last_name,last_update
0,1,PENELOPE,GUINESS,2020-12-23 07:12:29
1,2,NICK,WAHLBERG,2020-12-23 07:12:29
2,3,ED,CHASE,2020-12-23 07:12:29
3,4,JENNIFER,DAVIS,2020-12-23 07:12:29
4,5,JOHNNY,LOLLOBRIGIDA,2020-12-23 07:12:29


## DELETE Rows:

In [10]:
cursor = connection_db.execute("""
                          DELETE FROM actor
                          WHERE first_name = 'PENELOPE';
                                """)

tables = pd.read_sql("""
                        SELECT *
                        FROM actor
                        LIMIT 5;""", connection_db)
tables

,actor_id,first_name,last_name,last_update
0,2,NICK,WAHLBERG,2020-12-23 07:12:29
1,3,ED,CHASE,2020-12-23 07:12:29
2,4,JENNIFER,DAVIS,2020-12-23 07:12:29
3,5,JOHNNY,LOLLOBRIGIDA,2020-12-23 07:12:29
4,6,BETTE,NICHOLSON,2020-12-23 07:12:29


## UPDATE Table:

Change `first_name` from 'NICK' to 'Mick' for the record actor_id = 2

In [11]:
cursor = connection_db.execute("""
                          UPDATE actor
                          SET first_name = "Mick"
                          WHERE actor_id = 2;
                                """)

tables = pd.read_sql("""
                        SELECT *
                        FROM actor
                        LIMIT 5;""", connection_db)
tables

,actor_id,first_name,last_name,last_update
0,2,Mick,WAHLBERG,2026-04-15 14:08:28
1,3,ED,CHASE,2020-12-23 07:12:29
2,4,JENNIFER,DAVIS,2020-12-23 07:12:29
3,5,JOHNNY,LOLLOBRIGIDA,2020-12-23 07:12:29
4,6,BETTE,NICHOLSON,2020-12-23 07:12:29


# ALIASES:

Aliases can be used for both Columnnames and Tablenames.

In [12]:
tables = pd.read_sql("""
                        SELECT actor_id as id, first_name, last_name, last_update
                        FROM actor as A
                        LIMIT 5;""", connection_db)
tables

,id,first_name,last_name,last_update
0,2,Mick,WAHLBERG,2026-04-15 14:08:28
1,3,ED,CHASE,2020-12-23 07:12:29
2,4,JENNIFER,DAVIS,2020-12-23 07:12:29
3,5,JOHNNY,LOLLOBRIGIDA,2020-12-23 07:12:29
4,6,BETTE,NICHOLSON,2020-12-23 07:12:29


# Subqueries:

Subqueries are really important and can help achieve a lot things.

Let's start with a simple example. Let us say we want all film information about only films that are PG rated and by actor_id 1.

In [13]:
tables = pd.read_sql("""
                        SELECT *
                        FROM film
                        WHERE rating = "PG" AND film_id IN
                            (SELECT film_id
                             FROM film_actor
                             WHERE actor_id = 1)
                        ;""", connection_db)
print(tables.shape)
tables

(6, 13)


,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2020-12-23 07:12:31
1,506,LADY STAGE,A Beautiful Character Study of a Woman And a M...,2006,1,None,4,4.99,67,14.99,PG,"Trailers,Deleted Scenes,Behind the Scenes",2020-12-23 07:12:38
2,605,MULHOLLAND BEAST,A Awe-Inspiring Display of a Husband And a Squ...,2006,1,None,7,2.99,157,13.99,PG,"Trailers,Deleted Scenes,Behind the Scenes",2020-12-23 07:12:39
3,635,OKLAHOMA JUMANJI,A Thoughtful Drama of a Dentist And a Womanize...,2006,1,None,7,0.99,58,15.99,PG,Behind the Scenes,2020-12-23 07:12:40
4,832,SPLASH GUMP,A Taut Saga of a Crocodile And a Boat who must...,2006,1,None,5,0.99,175,16.99,PG,"Trailers,Commentaries,Deleted Scenes,Behind th...",2020-12-23 07:12:42
5,980,WIZARD COLDBLOODED,A Lacklusture Display of a Robot And a Girl wh...,2006,1,None,4,4.99,75,12.99,PG,"Commentaries,Deleted Scenes,Behind the Scenes",2020-12-23 07:12:44


Next we will use a subquery in the FROM statement.

Lets say we are interested to find the average `rental_rate` of films for each `rating` category, considering only those `rating` categories with an average rental rate of at least 3 euro.

In [14]:
tables = pd.read_sql("""
                             SELECT  rating, AVG(rental_rate) AS avg_rental_rate
                             FROM film
                             GROUP BY rating
                        ;""", connection_db)
print(tables.shape)
tables

(5, 2)


,rating,avg_rental_rate
0,G,2.888876
1,NC-17,2.970952
2,PG,3.051856
3,PG-13,3.034843
4,R,2.938718


In [15]:
tables = pd.read_sql("""
                        SELECT rating, avg_rental_rate
                        FROM
                            (SELECT  rating, AVG(rental_rate) AS avg_rental_rate
                             FROM film
                             GROUP BY rating)
                        WHERE avg_rental_rate>=3
                        ;""", connection_db)
print(tables.shape)
tables

(2, 2)


,rating,avg_rental_rate
0,PG,3.051856
1,PG-13,3.034843


Note: We could have achieved the same thing with HAVING! Thus a query for obtaining a particular purpose can be achieved in multiple ways.

In [16]:
tables = pd.read_sql("""
                             SELECT  rating, AVG(rental_rate) AS avg_rental_rate
                             FROM film
                             GROUP BY rating
                             HAVING avg_rental_rate>=3
                        ;""", connection_db)
print(tables.shape)
tables

(2, 2)


,rating,avg_rental_rate
0,PG,3.051856
1,PG-13,3.034843


Now let's look at multiple nested subqueries!

In [17]:
tables = pd.read_sql("""
                        SELECT *
                        FROM payment
                        ;""", connection_db)
print(tables.shape)
tables

(16049, 7)


,payment_id,customer_id,staff_id,rental_id,amount,payment_date,last_update
0,1,1,1,76.0,2.99,2005-05-25 11:30:37.000,2020-12-23 07:19:10
1,2,1,1,573.0,0.99,2005-05-28 10:35:23.000,2020-12-23 07:19:10
2,3,1,1,1185.0,5.99,2005-06-15 00:54:12.000,2020-12-23 07:19:10
3,4,1,2,1422.0,0.99,2005-06-15 18:02:53.000,2020-12-23 07:19:10
4,5,1,2,1476.0,9.99,2005-06-15 21:08:46.000,2020-12-23 07:19:10
...,...,...,...,...,...,...,...
16044,16045,599,1,14599.0,4.99,2005-08-21 17:43:42.000,2020-12-23 07:22:55
16045,16046,599,1,14719.0,1.99,2005-08-21 21:41:57.000,2020-12-23 07:22:55
16046,16047,599,2,15590.0,8.99,2005-08-23 06:09:44.000,2020-12-23 07:22:55
16047,16048,599,2,15719.0,2.99,2005-08-23 11:08:46.000,2020-12-23 07:22:55


In [18]:
tables = pd.read_sql("""
                        SELECT *
                        FROM rental
                        ;""", connection_db)
print(tables.shape)
tables

(16044, 7)


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30.000,367,130,2005-05-26 22:04:30.000,1,2020-12-23 07:15:20
1,2,2005-05-24 22:54:33.000,1525,459,2005-05-28 19:40:33.000,1,2020-12-23 07:15:20
2,3,2005-05-24 23:03:39.000,1711,408,2005-06-01 22:12:39.000,1,2020-12-23 07:15:20
3,4,2005-05-24 23:04:41.000,2452,333,2005-06-03 01:43:41.000,2,2020-12-23 07:15:20
4,5,2005-05-24 23:05:21.000,2079,222,2005-06-02 04:33:21.000,1,2020-12-23 07:15:20
...,...,...,...,...,...,...,...
16039,16045,2005-08-23 22:25:26.000,772,14,2005-08-25 23:54:26.000,1,2020-12-23 07:19:10
16040,16046,2005-08-23 22:26:47.000,4364,74,2005-08-27 18:02:47.000,2,2020-12-23 07:19:10
16041,16047,2005-08-23 22:42:48.000,2088,114,2005-08-25 02:48:48.000,2,2020-12-23 07:19:10
16042,16048,2005-08-23 22:43:07.000,2019,103,2005-08-31 21:33:07.000,1,2020-12-23 07:19:10


In [19]:
tables = pd.read_sql("""
                        SELECT *
                        FROM inventory
                        ;""", connection_db)
print(tables.shape)
tables

(4581, 4)


,inventory_id,film_id,store_id,last_update
0,1,1,1,2020-12-23 07:12:45
1,2,1,1,2020-12-23 07:12:45
2,3,1,1,2020-12-23 07:12:45
3,4,1,1,2020-12-23 07:12:45
4,5,1,2,2020-12-23 07:12:45
...,...,...,...,...
4576,4577,1000,1,2020-12-23 07:13:43
4577,4578,1000,2,2020-12-23 07:13:43
4578,4579,1000,2,2020-12-23 07:13:43
4579,4580,1000,2,2020-12-23 07:13:43


Find out customers and payment amounts where film_id = 1 was rented out.

In [61]:
tables = pd.read_sql("""
                        SELECT customer_id, amount
                        FROM payment
                        WHERE rental_id IN
                            (SELECT rental_id
                            FROM rental
                            WHERE inventory_id IN
                                (SELECT inventory_id
                                FROM inventory
                                WHERE film_id = 1))
                        ;""", connection_db)
print(tables.shape)
tables

(23, 2)


,customer_id,amount
0,8,0.99
1,34,0.99
2,39,0.99
3,44,0.99
4,92,0.99
5,161,0.99
6,170,0.99
7,252,0.99
8,279,3.99
9,301,0.99


Lastly, let's look at subquery to generate a new column in SELECT statement.

Find the number of payments that every staff member has processed. 

In [20]:
tables = pd.read_sql("""
                        SELECT *
                        FROM staff
                        ;""", connection_db)
print(tables.shape)
tables

(2, 11)


,staff_id,first_name,last_name,address_id,picture,email,store_id,active,username,password,last_update
0,1,Mike,Hillyer,3,None,Mike.Hillyer@sakilastaff.com,1,1,Mike,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31
1,2,Jon,Stephens,4,None,Jon.Stephens@sakilastaff.com,2,1,Jon,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31


In [21]:
tables = pd.read_sql("""
                        SELECT *
                        FROM payment
                        ;""", connection_db)
print(tables[['payment_id', 'staff_id']].isna().any())
tables.head()

payment_id    False
staff_id      False
dtype: bool


,payment_id,customer_id,staff_id,rental_id,amount,payment_date,last_update
0,1,1,1,76.0,2.99,2005-05-25 11:30:37.000,2020-12-23 07:19:10
1,2,1,1,573.0,0.99,2005-05-28 10:35:23.000,2020-12-23 07:19:10
2,3,1,1,1185.0,5.99,2005-06-15 00:54:12.000,2020-12-23 07:19:10
3,4,1,2,1422.0,0.99,2005-06-15 18:02:53.000,2020-12-23 07:19:10
4,5,1,2,1476.0,9.99,2005-06-15 21:08:46.000,2020-12-23 07:19:10


In [23]:
tables = pd.read_sql("""
                        SELECT first_name, last_name,
                          (SELECT COUNT(*) AS rent_sales
                           FROM payment
                           WHERE staff.staff_id = payment.staff_id)  AS rent_sales
                        FROM staff
                        ;""", connection_db)
print(tables.shape)
tables

(2, 3)


,first_name,last_name,rent_sales
0,Mike,Hillyer,8057
1,Jon,Stephens,7992


In [28]:
# alternatively
tables = pd.read_sql("""
                        SELECT staff.first_name, staff.last_name, COUNT(payment.payment_id) AS rent_sales
                        FROM staff
                        LEFT JOIN payment ON staff.staff_id = payment.staff_id
                        GROUP BY staff.staff_id, staff.first_name, staff.last_name  -- note that staff.staff_id is not listed in the SELECT clause
                        ;""", connection_db)
print(tables.shape)
tables

(2, 3)


,first_name,last_name,rent_sales
0,Mike,Hillyer,8057
1,Jon,Stephens,7992


# SET Operations:

This segment will deal with various `SET` operations.

In [29]:
tables = pd.read_sql("""
                        SELECT *
                        FROM film as A;""", connection_db)
tables.head(2)

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,None,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2020-12-23 07:12:31
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,None,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2020-12-23 07:12:31


In [30]:
tables = pd.read_sql("""
                        SELECT *
                        FROM film_actor as A;""", connection_db)
tables.head()

,actor_id,film_id,last_update
0,1,1,2020-12-23 07:13:43
1,1,23,2020-12-23 07:13:43
2,1,25,2020-12-23 07:13:43
3,1,106,2020-12-23 07:13:43
4,1,140,2020-12-23 07:13:43


We want to get all film id's where `rating` is 'PG' OR `actor_id` is 1

In [31]:
tables = pd.read_sql("""
                         SELECT film_id
                         FROM film
                         WHERE rating = "PG"
                         UNION 
                         SELECT film_id
                         FROM film_actor
                         WHERE actor_id = 1;""", connection_db)
print(tables.shape)
tables.head()

(207, 1)


,film_id
0,1
1,6
2,12
3,13
4,19


`UNION` keeps common `film_id` only once. We can display all occurences with `UNION ALL`

In [32]:
tables = pd.read_sql("""
                        SELECT film_id
                         FROM film
                         WHERE rating = "PG"
                         UNION ALL
                         SELECT film_id
                         FROM film_actor
                         WHERE actor_id = 1;""", connection_db)
print(tables.shape)
tables[tables.duplicated(keep=False)]  ## shows all duplicates 

(213, 1)


,film_id
0,1
78,506
103,605
111,635
156,832
189,980
194,1
204,506
206,605
207,635


In [33]:
## alternatively
tables = pd.read_sql("""
                        SELECT film_id
                         FROM 
                             (SELECT film_id
                              FROM film
                              WHERE rating = "PG"
                              UNION ALL
                              SELECT film_id
                              FROM film_actor
                              WHERE actor_id = 1)
                          WHERE film_id IN 
                            (SELECT film_id
                             FROM 
                             (SELECT film_id
                              FROM film
                              WHERE rating = "PG"
                              UNION ALL
                              SELECT film_id
                              FROM film_actor
                              WHERE actor_id = 1)
                             GROUP BY film_id
                             HAVING COUNT(film_id) > 1)
                          ;""", connection_db)
print(tables.shape)
tables

(12, 1)


,film_id
0,1
1,506
2,605
3,635
4,832
5,980
6,1
7,506
8,605
9,635


Now we are interested in only PG films by actor_id = 1. We will use 'INTERSECT'.

In [34]:
tables = pd.read_sql("""
                        SELECT film_id
                         FROM film
                         WHERE rating = "PG"
                         INTERSECT
                         SELECT film_id
                         FROM film_actor
                         WHERE actor_id = 1;""", connection_db)
print(tables.shape)
tables

(6, 1)


,film_id
0,1
1,506
2,605
3,635
4,832
5,980


Note: SQLite does not have INTERSECT ALL operator. 

Now we are interested PG films but not by actor_id = 1. We will use 'EXCEPT'.
Similarly, EXCEPT ALL does not exist in SQLite.

In [35]:
tables = pd.read_sql("""
                        SELECT film_id
                         FROM film
                         WHERE rating = "PG"
                         EXCEPT
                         SELECT film_id
                         FROM film_actor
                         WHERE actor_id = 1;""", connection_db)
print(tables.shape)
tables

(188, 1)


,film_id
0,6
1,12
2,13
3,19
4,37
...,...
183,966
184,983
185,985
186,987


In [36]:
tables = pd.read_sql("""
                        SELECT film_id
                         FROM film
                         WHERE rating = "PG";""", connection_db)
print(tables.shape)
tables.head()

(194, 1)


,film_id
0,1
1,6
2,12
3,13
4,19


# Joins

## Cartesian Product or Cross Join

In [37]:
tables = pd.read_sql("""
                        SELECT *
                        FROM staff
                        ;""", connection_db)
print(tables.shape)
tables

(2, 11)


,staff_id,first_name,last_name,address_id,picture,email,store_id,active,username,password,last_update
0,1,Mike,Hillyer,3,None,Mike.Hillyer@sakilastaff.com,1,1,Mike,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31
1,2,Jon,Stephens,4,None,Jon.Stephens@sakilastaff.com,2,1,Jon,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31


In [38]:
tables = pd.read_sql("""
                        SELECT *
                        FROM store
                        ;""", connection_db)
print(tables.shape)
tables

(2, 4)


,store_id,manager_staff_id,address_id,last_update
0,1,1,1,2020-12-23 07:12:31
1,2,2,2,2020-12-23 07:12:31


In [39]:
tables = pd.read_sql("""
                        SELECT *
                        FROM staff, store
                        ;""", connection_db)
print(tables.shape)
tables

(4, 15)


,staff_id,first_name,last_name,address_id,picture,email,store_id,active,username,password,last_update,store_id,manager_staff_id,address_id,last_update
0,1,Mike,Hillyer,3,None,Mike.Hillyer@sakilastaff.com,1,1,Mike,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31,1,1,1,2020-12-23 07:12:31
1,1,Mike,Hillyer,3,None,Mike.Hillyer@sakilastaff.com,1,1,Mike,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31,2,2,2,2020-12-23 07:12:31
2,2,Jon,Stephens,4,None,Jon.Stephens@sakilastaff.com,2,1,Jon,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31,1,1,1,2020-12-23 07:12:31
3,2,Jon,Stephens,4,None,Jon.Stephens@sakilastaff.com,2,1,Jon,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31,2,2,2,2020-12-23 07:12:31


In [40]:
tables = pd.read_sql("""
                        SELECT *
                        FROM staff CROSS JOIN store
                        ;""", connection_db)
print(tables.shape)
tables

(4, 15)


,staff_id,first_name,last_name,address_id,picture,email,store_id,active,username,password,last_update,store_id,manager_staff_id,address_id,last_update
0,1,Mike,Hillyer,3,None,Mike.Hillyer@sakilastaff.com,1,1,Mike,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31,1,1,1,2020-12-23 07:12:31
1,1,Mike,Hillyer,3,None,Mike.Hillyer@sakilastaff.com,1,1,Mike,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31,2,2,2,2020-12-23 07:12:31
2,2,Jon,Stephens,4,None,Jon.Stephens@sakilastaff.com,2,1,Jon,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31,1,1,1,2020-12-23 07:12:31
3,2,Jon,Stephens,4,None,Jon.Stephens@sakilastaff.com,2,1,Jon,8cb2237d0679ca88db6464eac60da96345513964,2020-12-23 07:12:31,2,2,2,2020-12-23 07:12:31


The 2 blocks of code produces same output!!

## INNER JOIN

In [41]:
tables = pd.read_sql("""
                        SELECT *
                        FROM film_category
                        ;""", connection_db)
print(tables.shape)
tables.head(3)

(1000, 3)


,film_id,category_id,last_update
0,1,6,2020-12-23 07:14:58
1,2,11,2020-12-23 07:14:58
2,3,6,2020-12-23 07:14:58


In [42]:
tables = pd.read_sql("""
                        SELECT *
                        FROM category
                        ;""", connection_db)
print(tables.shape)
tables.head(6)

(16, 3)


,category_id,name,last_update
0,1,Action,2020-12-23 07:12:31
1,2,Animation,2020-12-23 07:12:31
2,3,Children,2020-12-23 07:12:31
3,4,Classics,2020-12-23 07:12:31
4,5,Comedy,2020-12-23 07:12:31
5,6,Documentary,2020-12-23 07:12:31


Let's join the film_category table with the category table to display all categories besides the category_id

In [43]:
tables = pd.read_sql("""
                        SELECT A.*, B.name AS category
                        FROM film_category AS A
                        INNER JOIN category AS B
                        ON A.category_id = B.category_id
                        ;""", connection_db)
print(tables.shape)
tables.head()

(1000, 4)


,film_id,category_id,last_update,category
0,1,6,2020-12-23 07:14:58,Documentary
1,2,11,2020-12-23 07:14:58,Horror
2,3,6,2020-12-23 07:14:58,Documentary
3,4,11,2020-12-23 07:14:58,Horror
4,5,8,2020-12-23 07:14:58,Family


Joins can be merged with subquery

In [44]:
tables = pd.read_sql("""
                        SELECT A.*, B.total_film
                        FROM category AS A
                        INNER JOIN  (SELECT category_id, COUNT(film_id) AS total_film FROM film_category GROUP BY category_id) AS B
                        ON A.category_id = B.category_id
                        ;""", connection_db)
print(tables.shape)
tables.head()

(16, 4)


,category_id,name,last_update,total_film
0,1,Action,2020-12-23 07:12:31,64
1,2,Animation,2020-12-23 07:12:31,66
2,3,Children,2020-12-23 07:12:31,60
3,4,Classics,2020-12-23 07:12:31,57
4,5,Comedy,2020-12-23 07:12:31,58


## LEFT Join

We want to find total payment amount of each customer from payment table, and also want to check whether the number of rentals for each customer is the same according to the data in the payment table and the rental table.

In [45]:
tables = pd.read_sql("""
                       SELECT *
                        FROM payment AS p
                        ;""", connection_db)
print(tables.shape)
tables.head(3)

(16049, 7)


,payment_id,customer_id,staff_id,rental_id,amount,payment_date,last_update
0,1,1,1,76.0,2.99,2005-05-25 11:30:37.000,2020-12-23 07:19:10
1,2,1,1,573.0,0.99,2005-05-28 10:35:23.000,2020-12-23 07:19:10
2,3,1,1,1185.0,5.99,2005-06-15 00:54:12.000,2020-12-23 07:19:10


In [46]:
tables = pd.read_sql("""
                       SELECT p.customer_id, p.rental_id, p.amount
                        FROM payment AS p
                        WHERE p.rental_id IS NULL OR p.customer_id IS NULL
                        ;""", connection_db)
print(tables.shape)
tables

(5, 3)


,customer_id,rental_id,amount
0,16,None,1.99
1,259,None,1.99
2,401,None,0.99
3,546,None,3.99
4,577,None,0.99


In [47]:
tables = pd.read_sql("""
                       SELECT *
                        FROM rental 
                        ;""", connection_db)
print(tables.shape)
tables.head(3)

(16044, 7)


,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30.000,367,130,2005-05-26 22:04:30.000,1,2020-12-23 07:15:20
1,2,2005-05-24 22:54:33.000,1525,459,2005-05-28 19:40:33.000,1,2020-12-23 07:15:20
2,3,2005-05-24 23:03:39.000,1711,408,2005-06-01 22:12:39.000,1,2020-12-23 07:15:20


In [48]:
tables = pd.read_sql("""
                       SELECT r.customer_id, r.rental_id, r.inventory_id
                        FROM rental AS r
                        WHERE r.rental_id IS NULL OR r.customer_id IS NULL
                        ;""", connection_db)
print(tables.shape)
tables.head()

(0, 3)


,customer_id,rental_id,inventory_id


In [49]:
tables = pd.read_sql("""
                       SELECT
                          r.customer_id     
                          , COUNT(p.rental_id) as total_rentals
                          , SUM(p.amount) AS total_amount
                          , COUNT (r.rental_id) as total_rental_check
                        FROM payment AS p   -- payment table is the left table
                        LEFT JOIN rental AS r
                        ON p.rental_id = r.rental_id
                        GROUP BY r.customer_id  
                        ORDER BY total_rentals DESC
                        ;""", connection_db)
print(tables.shape)
tables.tail()

(600, 4)


,customer_id,total_rentals,total_amount,total_rental_check
595,281.0,14,50.86,14
596,110.0,14,59.86,14
597,61.0,14,58.86,14
598,318.0,12,52.88,12
599,NaN,0,9.95,0


In [50]:
tables = pd.read_sql("""
                       SELECT
                          r.customer_id
                          , COUNT(p.rental_id) as total_rentals
                          , SUM(p.amount) AS total_amount
                          , COUNT (r.rental_id) as total_rental_check
                        FROM rental AS r  -- rental table is the left table
                        LEFT JOIN payment AS p
                        ON p.rental_id = r.rental_id
                        GROUP BY r.customer_id
                        ORDER BY total_rentals DESC
                        ;""", connection_db)
print(tables.shape)
tables.tail(4)

(599, 4)


,customer_id,total_rentals,total_amount,total_rental_check
595,61,14,58.86,14
596,110,14,59.86,14
597,281,14,50.86,14
598,318,12,52.88,12


## Full Outer Join

Full Outer Join Doesn't Exist! 

## After completing all the tasks above

In [51]:
# if renaming the table 'movie_language' back to 'language'
connection_db.execute('''ALTER TABLE movie_language
                         RENAME TO language;
                         ''')
tables = pd.read_sql("""SELECT *
                        FROM sqlite_master
                        WHERE type='table';""", connection_db)
tables

,type,name,tbl_name,rootpage,sql
0,table,actor,actor,2,CREATE TABLE actor (\n actor_id numeric NOT N...
1,table,country,country,5,CREATE TABLE country (\n country_id SMALLINT ...
2,table,city,city,9,"CREATE TABLE city (\n city_id int NOT NULL,\n..."
3,table,address,address,13,CREATE TABLE address (\n address_id int NOT N...
4,table,language,language,17,"CREATE TABLE ""language"" (\n language_id SMALL..."
5,table,category,category,19,CREATE TABLE category (\n category_id SMALLIN...
6,table,customer,customer,22,CREATE TABLE customer (\n customer_id INT NOT...
7,table,film,film,29,"CREATE TABLE film (\n film_id int NOT NULL,\n..."
8,table,film_actor,film_actor,35,CREATE TABLE film_actor (\n actor_id INT NOT ...
9,table,film_category,film_category,40,CREATE TABLE film_category (\n film_id INT NO...


In [52]:
# close the connection
connection_db.close() 